In [4]:
# 1 
import requests
import json


url = "https://api.binance.com/api/v3/ticker/24hr"

try:
    # Send GET request
    response = requests.get(url, timeout=10)

    # Check if the request was successful
    response.raise_for_status()

    # Convert response to JSON
    data = response.json()

    # List of 10 popular cryptocurrencies
    popular_coins = [
        "BTCUSDT",
        "ETHUSDT",
        "BNBUSDT",
        "XRPUSDT",
        "SOLUSDT",
        "ADAUSDT",
        "DOGEUSDT",
        "TRXUSDT",
        "DOTUSDT",
        "LTCUSDT"
    ]

    # Filter data for the selected coins
    crypto_data = [
        coin for coin in data
        if coin["symbol"] in popular_coins
    ]

    # Save raw JSON response to a file
    with open("crypto_data.json", "w", encoding="utf-8") as file:
        json.dump(crypto_data, file, indent=4)

    print("Data successfully saved to 'crypto_data.json'")

except requests.exceptions.HTTPError as err:
    print("HTTP Error:", err)

except requests.exceptions.ConnectionError:
    print("Error: Unable to connect to the Binance API.")

except requests.exceptions.Timeout:
    print("Error: The request timed out.")

except requests.exceptions.RequestException as err:
    print("Request Error:", err)


c:\Users\skadi\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Data successfully saved to 'crypto_data.json'


In [5]:
# 2
def find_most_volatile_coin(data):
    """
    Returns the symbol of the coin with the highest
    absolute percentage price change in the last 24 hours.
    """

    most_volatile = max(
        data,
        key=lambda coin: abs(float(coin["priceChangePercent"]))
    )

    return most_volatile["symbol"]

In [6]:
# 3
import json

# Load data from crypto_data.json
with open("crypto_data.json", "r", encoding="utf-8") as file:
    crypto_data = json.load(file)

# Calculate the average price
total_price = 0

for coin in crypto_data:
    total_price += float(coin["lastPrice"])

average_price = total_price / len(crypto_data)

print(f"Average Price of All Coins: {average_price:.2f}\n")

# Print coins trading below the average price
print("Coins Trading Below the Average Price:")
print("-" * 40)

found = False

for coin in crypto_data:
    price = float(coin["lastPrice"])

    if price < average_price:
        print(f"{coin['symbol']} : ${price:.2f}")
        found = True

if not found:
    print("No coins are trading below the average price.")

Average Price of All Coins: 6227.75

Coins Trading Below the Average Price:
----------------------------------------
ETHUSDT : $1580.16
BNBUSDT : $553.60
LTCUSDT : $42.66
ADAUSDT : $0.15
XRPUSDT : $1.05
TRXUSDT : $0.32
DOGEUSDT : $0.07
SOLUSDT : $72.72
DOTUSDT : $0.82


In [7]:
# 4
import json

def rank_coins_by_volume(data):
    """
    Sort coins by total traded volume (descending)
    and print top 5 with rank and volume.
    """

    # Sort coins by volume in descending order
    sorted_coins = sorted(
        data,
        key=lambda coin: float(coin["volume"]),
        reverse=True
    )

    print("\nTop 5 Coins by Volume")
    print("-" * 35)

    # Print top 5 coins
    for rank, coin in enumerate(sorted_coins[:5], start=1):
        print(f"{rank}. {coin['symbol']} -> Volume: {coin['volume']}")

    return sorted_coins[:5]


# Example usage
if __name__ == "__main__":
    with open("crypto_data.json", "r", encoding="utf-8") as file:
        crypto_data = json.load(file)

    rank_coins_by_volume(crypto_data)


Top 5 Coins by Volume
-----------------------------------
1. DOGEUSDT -> Volume: 343228367.00000000
2. TRXUSDT -> Volume: 94589547.80000000
3. ADAUSDT -> Volume: 75869421.90000000
4. XRPUSDT -> Volume: 65546545.50000000
5. DOTUSDT -> Volume: 6515897.18000000


In [ ]:
# 5

import requests
import json
import time
import schedule

URL = "https://api.binance.com/api/v3/ticker/24hr"

POPULAR_COINS = {
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "SOLUSDT",
    "ADAUSDT", "DOGEUSDT", "TRXUSDT", "DOTUSDT", "LTCUSDT"
}


def fetch_binance_data(max_retries=5):
    """
    Fetch Binance 24h data with exponential backoff
    and rate limit handling (HTTP 429).
    """

    delay = 2  # initial backoff delay

    for attempt in range(max_retries):
        try:
            response = requests.get(URL, timeout=10)

            # Handle rate limiting
            if response.status_code == 429:
                print(f"Rate limited (429). Retrying in {delay} seconds...")
                time.sleep(delay)
                delay *= 2
                continue

            response.raise_for_status()

            data = response.json()

            # Filter only selected coins
            filtered = [
                coin for coin in data
                if coin["symbol"] in POPULAR_COINS
            ]

            return filtered

        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2

    print("Failed after maximum retries.")
    return []


def save_data(data):
    with open("crypto_data.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def analyze(data):
    if not data:
        print("No data to analyze.")
        return

    print("\n===== CRYPTO ANALYSIS =====")

    # Most volatile coin
    most_volatile = max(
        data,
        key=lambda c: abs(float(c["priceChangePercent"]))
    )

    print("Most Volatile Coin:", most_volatile["symbol"])

    # Average price
    prices = [float(c["lastPrice"]) for c in data]
    avg_price = sum(prices) / len(prices)

    print(f"Average Price: {avg_price:.4f}")

    print("\nCoins below average price:")
    for c in data:
        if float(c["lastPrice"]) < avg_price:
            print("-", c["symbol"])


def job():
    print("\nFetching Binance data...")

    data = fetch_binance_data()

    if data:
        save_data(data)
        analyze(data)


# Run once immediately
job()

# Schedule every hour
schedule.every().hour.do(job)

print("\nScheduler started... Running every hour...\n")

while True:
    schedule.run_pending()
    time.sleep(1)